In [9]:
import pandas as pd
import numpy as np

# Read the Excel file
df = pd.read_excel('first-100-rows.xlsx')

print("=== Sampling Rate Confirmation ===")
print(f"Column names: {df.columns.tolist()}")
print(f"First column (time): {df.iloc[:5, 0].values}")

# Calculate sampling rate from timestamp differences
timestamps = df.iloc[:, 0].values
diff = timestamps[1] - timestamps[0]
sfreq = 1.0 / diff
print(f"Time difference between samples: {diff:.10f} seconds")
print(f"Sampling rate: {sfreq:.2f} Hz")

=== Sampling Rate Confirmation ===
Column names: ['Unnamed: 0', 'FP1', 'FP2', 'F7', 'F3', 'FZ', 'F4', 'F8', 'T7', 'C3', 'CZ', 'C4', 'T8', 'P7', 'P3', 'PZ', 'P4', 'P8', 'O1', 'O2', 'A1', 'A2', 'EMG1', 'EMG2', 'ECG1', 'ECG2']
First column (time): [0.         0.00048828 0.00097656 0.00146484 0.00195312]
Time difference between samples: 0.0004882812 seconds
Sampling rate: 2048.00 Hz


In [8]:
!pip3 install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
# Try reading as text first
try:
    with open('MI.evt', 'r') as f:
        content = f.read()
    print("=== EVT File (text) ===")
    print(content[:2000])
except:
    print("Not a text file, trying binary...")

# Try reading as binary
with open('MI.evt', 'rb') as f:
    raw_bytes = f.read()
print(f"\nFile size: {len(raw_bytes)} bytes")
print(f"First 200 bytes (hex):")
print(raw_bytes[:200].hex())
print(f"\nFirst 200 bytes (attempted ASCII):")
print(raw_bytes[:200])

=== EVT File (text) ===
number	type	latency	urevent	 duration

File size: 37 bytes
First 200 bytes (hex):
6e756d6265720974797065096c6174656e63790975726576656e7409206475726174696f6e

First 200 bytes (attempted ASCII):
b'number\ttype\tlatency\turevent\t duration'


In [6]:
import pandas as pd

# Read annotation file
df_ann = pd.read_csv(
    'Annotation.txt',
    sep='\t'
)

print("=== Full Annotation File ===")
print(df_ann)
print(f"\nTotal annotations: {len(df_ann)}")

# Count motor imagery events
mi_labels = ['l', 'r', 'L', 'R']
both_labels = ['b', 'B']
rest_labels = ['s', 'S']

df_ann['Name'] = df_ann['Name'].astype(str).str.strip()

print(f"\nLeft hand trials:  {len(df_ann[df_ann['Name'].isin(['l', 'L'])])}")
print(f"Right hand trials: {len(df_ann[df_ann['Name'].isin(['r', 'R'])])}")
print(f"Both hands trials: {len(df_ann[df_ann['Name'].isin(both_labels)])}")
print(f"Rest/idle trials:  {len(df_ann[df_ann['Name'].isin(rest_labels)])}")
print(f"\nAll unique labels: {df_ann['Name'].unique()}")

=== Full Annotation File ===
    number                                Name   latency  urevent  duration
0        1  Date: 2020-3-5  TimeStart- 17:11:9     0.000        1      -1.0
1        2                 2020_03_05_17_02_44     0.000        2      -1.0
2        3                                 NaN     0.500        3      -1.0
3        4                           Eyes Open     5.890        4      -1.0
4        5                          Eyes Close    23.064        5      -1.0
..     ...                                 ...       ...      ...       ...
62      63                                   S  1306.715       63      -1.0
63      64                           Eyes Open  1332.549       64      -1.0
64      65                          Eyes Close  1339.399       65      -1.0
65      66                                   R  1350.019       66      -1.0
66      67                           Eyes Open  1362.196       67      -1.0

[67 rows x 5 columns]

Total annotations: 67

Left hand tr

In [11]:
import mne
import os

mne.set_log_level('WARNING')

# Fix path - go up one level from notebooks/
raw = mne.io.read_raw_gdf(
    '../data/bci_iv_2a/BCICIV_2a_gdf/A01T.gdf',
    preload=False
)

print("BCI IV-2a actual channel names from MNE:")
for i, ch in enumerate(raw.ch_names):
    print(f"  {i:2d}: {ch}")

print(f"\nTotal channels: {len(raw.ch_names)}")
print(f"Sampling rate:  {raw.info['sfreq']} Hz")

BCI IV-2a actual channel names from MNE:
   0: EEG-Fz
   1: EEG-0
   2: EEG-1
   3: EEG-2
   4: EEG-3
   5: EEG-4
   6: EEG-5
   7: EEG-C3
   8: EEG-6
   9: EEG-Cz
  10: EEG-7
  11: EEG-C4
  12: EEG-8
  13: EEG-9
  14: EEG-10
  15: EEG-11
  16: EEG-12
  17: EEG-13
  18: EEG-14
  19: EEG-Pz
  20: EEG-15
  21: EEG-16
  22: EOG-left
  23: EOG-central
  24: EOG-right

Total channels: 25
Sampling rate:  250.0 Hz


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


In [12]:
import mne
mne.set_log_level('WARNING')

# Step 1: Load BCI IV-2a
raw = mne.io.read_raw_gdf(
    '../data/bci_iv_2a/BCICIV_2a_gdf/A01T.gdf',
    preload=False
)

# Step 2: Apply our channel mapping
BCI_CHANNEL_MAP = {
    'EEG-Fz':  'FZ',
    'EEG-C3':  'C3',
    'EEG-Cz':  'CZ',
    'EEG-C4':  'C4',
    'EEG-Pz':  'PZ',
    'EEG-0':   'FC3',
    'EEG-1':   'FC1',
    'EEG-2':   'FCZ',
    'EEG-3':   'FC2',
    'EEG-4':   'FC4',
    'EEG-5':   'C5',
    'EEG-6':   'C1',
    'EEG-7':   'C2',
    'EEG-8':   'C6',
    'EEG-9':   'CP3',
    'EEG-10':  'CP1',
    'EEG-11':  'CPZ',
    'EEG-12':  'CP2',
    'EEG-13':  'CP4',
    'EEG-14':  'P1',
    'EEG-15':  'P2',
    'EEG-16':  'POZ',
    'EOG-left':    'EOG-LEFT',
    'EOG-central': 'EOG-CENTRAL',
    'EOG-right':   'EOG-RIGHT',
}

raw.rename_channels(BCI_CHANNEL_MAP)

# Step 3: Print renamed channels
print("Renamed BCI IV-2a channels:")
for i, ch in enumerate(raw.ch_names):
    print(f"  {i:2d}: {ch}")

# Step 4: Check our 5 common channels exist
COMMON_CHANNELS = ['FZ', 'C3', 'CZ', 'C4', 'PZ']
print(f"\nCommon channels present:")
for ch in COMMON_CHANNELS:
    present = ch in raw.ch_names
    print(f"  {ch}: {'YES' if present else 'MISSING'}")

# Step 5: Official montage order for comparison
OFFICIAL_ORDER = [
    'FZ', 'FC3', 'FC1', 'FCZ', 'FC2', 'FC4',
    'C5', 'C3', 'C1', 'CZ', 'C2', 'C4', 'C6',
    'CP3', 'CP1', 'CPZ', 'CP2', 'CP4',
    'P1', 'PZ', 'P2', 'POZ'
]

eeg_channels = [
    ch for ch in raw.ch_names
    if not ch.startswith('EOG')
]

print(f"\nOur renamed order matches official:")
for i, (ours, official) in enumerate(
    zip(eeg_channels, OFFICIAL_ORDER)
):
    match = 'OK' if ours == official else 'MISMATCH'
    print(f"  {i:2d}: {ours:6s} vs {official:6s} {match}")

Renamed BCI IV-2a channels:
   0: FZ
   1: FC3
   2: FC1
   3: FCZ
   4: FC2
   5: FC4
   6: C5
   7: C3
   8: C1
   9: CZ
  10: C2
  11: C4
  12: C6
  13: CP3
  14: CP1
  15: CPZ
  16: CP2
  17: CP4
  18: P1
  19: PZ
  20: P2
  21: POZ
  22: EOG-LEFT
  23: EOG-CENTRAL
  24: EOG-RIGHT

Common channels present:
  FZ: YES
  C3: YES
  CZ: YES
  C4: YES
  PZ: YES

Our renamed order matches official:
   0: FZ     vs FZ     OK
   1: FC3    vs FC3    OK
   2: FC1    vs FC1    OK
   3: FCZ    vs FCZ    OK
   4: FC2    vs FC2    OK
   5: FC4    vs FC4    OK
   6: C5     vs C5     OK
   7: C3     vs C3     OK
   8: C1     vs C1     OK
   9: CZ     vs CZ     OK
  10: C2     vs C2     OK
  11: C4     vs C4     OK
  12: C6     vs C6     OK
  13: CP3    vs CP3    OK
  14: CP1    vs CP1    OK
  15: CPZ    vs CPZ    OK
  16: CP2    vs CP2    OK
  17: CP4    vs CP4    OK
  18: P1     vs P1     OK
  19: PZ     vs PZ     OK
  20: P2     vs P2     OK
  21: POZ    vs POZ    OK


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
